# Pseudobulk Differential Expression

**REQUIRED DAY 3**

## Same question, two ways

The question: does IFN-β stimulation change gene expression in **CD14+ Monocytes** (the largest monocyte population, and one of the most IFN-responsive cell types)? We'll answer it twice — the way that treats every cell as an independent replicate (wrong, but what a naive script often does by default), and the way that respects the donor as the actual independent unit — and compare the real results.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read_h5ad("/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad")
CELL_TYPE = "CD14+ Monocytes"
sub = adata[adata.obs["cell_type"] == CELL_TYPE].copy()
print(sub.n_obs, "cells")
sub.obs.groupby(["replicate", "label"]).size().unstack()

5,697 cells, unevenly split across 8 donors and 2 conditions (some donors have far more monocytes than others) — exactly the kind of imbalance that makes per-cell testing risky: donors with more cells dominate the naive test without actually representing more biological replicates.

## Approach 1: naive per-cell test (the wrong way, run anyway to see what happens)

In [ ]:
sub.layers["counts"] = sub.X.copy()
sc.pp.normalize_total(sub, target_sum=1e4)
sc.pp.log1p(sub)
sc.tl.rank_genes_groups(sub, groupby="label", groups=["stim"], reference="ctrl", method="wilcoxon")
naive_results = sc.get.rank_genes_groups_df(sub, group="stim")
naive_sig = set(naive_results.loc[naive_results["pvals_adj"] < 0.05, "names"])
print(f"{len(naive_sig)} / {len(naive_results)} genes significant at padj<0.05")

This treats all 5,697 cells as if they were 5,697 independent measurements of the stim-vs-ctrl effect. They aren't — they're 8 donors' worth of cells, unevenly sampled.

## Approach 2: pseudobulk (the correct way)

Aggregate raw counts to one profile per donor per condition, then test with `pydeseq2` using a paired design (`~replicate + label`) that explicitly models donor-to-donor baseline differences before testing the condition effect.

In [ ]:
raw = adata[adata.obs["cell_type"] == CELL_TYPE]
counts_df = pd.DataFrame(
    raw.X.toarray() if hasattr(raw.X, "toarray") else raw.X,
    index=raw.obs_names, columns=raw.var_names,
)
counts_df["replicate"] = raw.obs["replicate"].values
counts_df["label"] = raw.obs["label"].values
pseudobulk = counts_df.groupby(["replicate", "label"]).sum(numeric_only=True)
pseudobulk.shape  # (16 samples, n_genes) -- 8 donors x 2 conditions

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

meta = pseudobulk.index.to_frame(index=False)
meta.index = [f"{r}_{c}" for r, c in zip(meta["replicate"], meta["label"])]
counts_mat = pseudobulk.reset_index(drop=True)
counts_mat.index = meta.index

# drop genes with fewer than 10 total counts across all 16 pseudobulk samples
keep = counts_mat.sum(axis=0) >= 10
counts_mat = counts_mat.loc[:, keep]
print("testing", counts_mat.shape[1], "genes after low-count filter")

# n_cpus set explicitly -- do not leave this as the default on a shared machine
dds = DeseqDataSet(counts=counts_mat.astype(int), metadata=meta.set_index(meta.index),
                   design="~replicate + label", n_cpus=4)
dds.deseq2()
stat_res = DeseqStats(dds, contrast=["label", "stim", "ctrl"], n_cpus=4)
stat_res.summary()
pb_results = stat_res.results_df
pb_sig = set(pb_results.loc[pb_results["padj"] < 0.05].index)
print(f"\n{len(pb_sig)} / {len(pb_results)} genes significant at padj<0.05")

## The real comparison

On this exact data: the naive test found **1,692 / 15,706** genes significant; pseudobulk found **3,411 / 10,086** (after the low-count filter). That's not the story you might expect — **pseudobulk found *more* significant genes, not fewer.**

That's the opposite of the usual pseudoreplication warning ("naive testing inflates false positives"), and it's worth sitting with *why*: `~replicate + label` explicitly removes donor-to-donor baseline variation before testing the condition effect — the same reason a paired t-test has more power than an unpaired one when subjects vary a lot at baseline. The naive test pools cells across donors with different baseline monocyte states, and that between-donor noise drowns out some real, consistent within-donor effects.

**The lesson is not "pseudobulk always finds more." It's that the naive test's p-values aren't trustworthy in *either* direction**, because its independence assumption is simply false — sometimes that inflates significance, sometimes it masks real signal, and you can't tell which without running the correctly-specified test.

In [ ]:
both = naive_sig & pb_sig
naive_only = naive_sig - pb_sig
pb_only = pb_sig - naive_sig
print(f"agree: {len(both)}, naive-only: {len(naive_only)}, pseudobulk-only: {len(pb_only)}")

Real numbers: 1,448 genes agree, 244 are naive-only (plausible false positives — significant only when donor structure is ignored), and 1,963 are pseudobulk-only (real, consistent within-donor effects the naive test's between-donor noise obscured).

## A sanity check that should reassure you

The most famous interferon-stimulated genes (ISG15, IFIT1, IFIT3, MX1, OAS1, STAT1) should show up as significant no matter how you test — their effect sizes are enormous. Check:

In [ ]:
for g in ["ISG15", "IFIT1", "IFIT3", "MX1", "OAS1", "STAT1"]:
    if g in pb_results.index:
        print(g, "padj =", pb_results.loc[g, "padj"], "| in pseudobulk sig:", g in pb_sig, "| in naive sig:", g in naive_sig)

All six should come back significant in both tests — when an effect is large enough, the naive test's flawed assumptions don't matter. The disagreement lives in the *marginal* genes, which is exactly where getting the statistics right or wrong actually changes your conclusion.

## Save your result

Lesson 06 (pathway analysis) and lesson 09 (visualization capstone) both read this back in, so every notebook can run on its own without depending on this session's in-memory state.

In [ ]:
import os

os.makedirs("results", exist_ok=True)
pb_results.to_csv("results/de_results_cd14mono.csv")
print("Saved to results/de_results_cd14mono.csv")


## Agent-assisted DE, done right

> Weak: "Run differential expression between stim and ctrl in CD14+ Monocytes."
>
> Strong: "Run differential expression between stim and ctrl in CD14+ Monocytes. Aggregate raw counts to one profile per donor per condition first — do not test at the cell level. Use a paired design across the 8 donors, and tell me how many genes you tested and what correction you applied before calling anything significant."

## Practice

Open a fresh Agent B session and run checklist items 19 (pseudobulk before testing), 20 (correction matches the actual test count), and 22 (paired structure preserved) against this notebook's own code and output.

## Further reading

- [pydeseq2 documentation](https://pydeseq2.readthedocs.io/)
- [Single-cell best practices — Differential gene expression analysis](https://www.sc-best-practices.org/conditions/differential_gene_expression.html)
- [Squair et al. 2021, "Confronting false discoveries in single-cell differential expression," Nature Communications](https://www.nature.com/articles/s41467-021-25960-2) — a good deeper read on exactly this problem: ~90% of published scRNA-seq DE studies used methods prone to these false discoveries.